### Imports


In [ ]:
import os
import kagglehub
import pandas as pd
import torch
import torch.nn as nn
import time
import copy
import random
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from PIL import Image
from math import floor
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import precision_recall_fscore_support

# 1. Data Preparation & Sanity Checks

In [ ]:
# Download dataset
path = kagglehub.dataset_download("imonbilk/industry-biscuit-cookie-dataset")
dataset_root = os.path.join(path, 'IndustryBiscuit')
images_path = os.path.join(dataset_root, 'Images')
csv_path = os.path.join(dataset_root, 'Annotations.csv')

dsPath = './FOLDER2'
# ... (Dataset folder structure logic from previous cell veixcHHfd6el) ...

Using Colab cache for faster access to the 'industry-biscuit-cookie-dataset' dataset.


In [ ]:
# Download dataset
path = kagglehub.dataset_download("imonbilk/industry-biscuit-cookie-dataset")

# Map exact absolute paths inside the downloaded folder
dataset_root = os.path.join(path, 'IndustryBiscuit')
images_path = os.path.join(dataset_root, 'Images')
csv_path = os.path.join(dataset_root, 'Annotations.csv')

print("Images source:", images_path)
print("CSV label path:", csv_path)

Using Colab cache for faster access to the 'industry-biscuit-cookie-dataset' dataset.
Images source: /kaggle/input/industry-biscuit-cookie-dataset/IndustryBiscuit/Images
CSV label path: /kaggle/input/industry-biscuit-cookie-dataset/IndustryBiscuit/Annotations.csv


In [ ]:
import shutil
import os

# Force destination path to remain inside Colab's visible /content workspace directory
dsPath = './FOLDER2'

# Remove existing folder to ensure a fresh split
if os.path.exists(dsPath):
    shutil.rmtree(dsPath)

# Rebalanced splits (approx 60% Train, 20% Valid, 20% Test)
# Total OK = 1896, Total NOK = 3004
nTrain_ok = 1137
nTrain_nok = 1802
nValid_ok = 379
nValid_nok = 601
nTest_ok = 380
nTest_nok = 601

rNComplete, rSObject, rCDefect = 0.4, 0.3, 0.3
cTrain_ok = cValid_ok = cTest_ok = 0
cTrainNC_nok = cValidNC_nok = cTestNC_nok = 0
cTrainSO_nok = cValidSO_nok = cTestSO_nok = 0
cTrainCD_nok = cValidCD_nok = cTestCD_nok = 0

nTrNC = floor(nTrain_nok * rNComplete)
nVaNC = floor(nValid_nok * rNComplete)
nTeNC = floor(nTest_nok * rNComplete)

nTrSO = floor(nTrain_nok * rSObject)
nVaSO = floor(nValid_nok * rSObject)
nTeSO = floor(nTest_nok * rSObject)

nTrCD = floor(nTrain_nok * rCDefect)
nVaCD = floor(nValid_nok * rCDefect)
nTeCD = floor(nTest_nok * rCDefect)

if not os.path.exists(dsPath):
    os.makedirs(dsPath + '/train/ok', exist_ok=True)
    os.makedirs(dsPath + '/train/nok', exist_ok=True)
    os.makedirs(dsPath + '/valid/ok', exist_ok=True)
    os.makedirs(dsPath + '/valid/nok', exist_ok=True)
    os.makedirs(dsPath + '/test/ok', exist_ok=True)
    os.makedirs(dsPath + '/test/nok', exist_ok=True)

    data = pd.read_csv(csv_path, usecols=['file', 'classDescription'])
    augm = 1226

    for key in range(1, 1226):
        for temp in range(0, 4):
            index = key if temp == 0 else augm
            if temp != 0: augm += 1

            value = data.iloc[index - 1, :]
            im = Image.open(os.path.join(images_path, value[0]))

            if value[1] == "Defect_No":
                if (cTrain_ok < nTrain_ok):
                    im.save(os.path.join(dsPath + '/train/ok', value[0]), format='jpeg')
                    cTrain_ok += 1
                elif (cValid_ok < nValid_ok):
                    im.save(os.path.join(dsPath + '/valid/ok', value[0]), format='jpeg')
                    cValid_ok += 1
                elif (cTest_ok < nTest_ok):
                    im.save(os.path.join(dsPath + '/test/ok', value[0]), format='jpeg')
                    cTest_ok += 1
            elif value[1] == "Defect_Shape":
                if (cTrainNC_nok < nTrNC):
                    im.save(os.path.join(dsPath + '/train/nok', value[0]), format='jpeg')
                    cTrainNC_nok += 1
                elif (cValidNC_nok < nVaNC):
                    im.save(os.path.join(dsPath + '/valid/nok', value[0]), format='jpeg')
                    cValidNC_nok += 1
                elif (cTestNC_nok < nTeNC):
                    im.save(os.path.join(dsPath + '/test/nok', value[0]), format='jpeg')
                    cTestNC_nok += 1
            elif value[1] == "Defect_Object":
                if (cTrainSO_nok < nTrSO):
                    im.save(os.path.join(dsPath + '/train/nok', value[0]), format='jpeg')
                    cTrainSO_nok += 1
                elif (cValidSO_nok < nVaSO):
                    im.save(os.path.join(dsPath + '/valid/nok', value[0]), format='jpeg')
                    cValidSO_nok += 1
                elif (cTestSO_nok < nTeSO):
                    im.save(os.path.join(dsPath + '/test/nok', value[0]), format='jpeg')
                    cTestSO_nok += 1
            elif value[1] == "Defect_Color":
                if (cTrainCD_nok < nTrCD):
                    im.save(os.path.join(dsPath + '/train/nok', value[0]), format='jpeg')
                    cTrainCD_nok += 1
                elif (cValidCD_nok < nVaCD):
                    im.save(os.path.join(dsPath + '/valid/nok', value[0]), format='jpeg')
                    cValidCD_nok += 1
                elif (cTestCD_nok < nTeCD):
                    im.save(os.path.join(dsPath + '/test/nok', value[0]), format='jpeg')
                    cTestCD_nok += 1
    print("Dataset folder structure initialized inside workspace.")
else:
    print("Folder structure already exists.")

/tmp/ipykernel_5415/1740631528.py:55: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  im = Image.open(os.path.join(images_path, value[0]))
/tmp/ipykernel_5415/1740631528.py:57: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if value[1] == "Defect_No":
/tmp/ipykernel_5415/1740631528.py:67: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  elif value[1] == "Defect_Shape":
/tmp/ipykernel_5415/1740631528.py:69: FutureWarning: Series.__getitem_

Dataset folder structure initialized inside workspace.


In [ ]:
# 1. Transform Pipeline
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 2. Point to the parent 'train' folder, NOT 'train/ok'
train_dataset = datasets.ImageFolder(root='./FOLDER2/train', transform=transform)

# 3. Create DataLoader
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# 4. Verify batch output dimensions
images, labels = next(iter(train_loader))
print("Batch image tensor shape:", images.shape) # Expected target shape format: [32, 3, 224, 224]
print("Class labels extracted mapping:", train_dataset.class_to_idx)

Batch image tensor shape: torch.Size([32, 3, 224, 224])
Class labels extracted mapping: {'nok': 0, 'ok': 1}


In [ ]:

# 2. Point to the parent 'train' folder, NOT 'train/ok'
test_dataset = datasets.ImageFolder(root='./FOLDER2/test', transform=transform)

# 3. Create DataLoader
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)


valid_dataset = datasets.ImageFolder(root='./FOLDER2/valid', transform=transform)
valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False)


# 4. Verify batch output dimensions
images, labels = next(iter(test_loader))
print("Batch image tensor shape:", images.shape) # Expected target shape format: [32, 3, 224, 224]
print("Class labels extracted mapping:", test_dataset.class_to_idx)

Batch image tensor shape: torch.Size([32, 3, 224, 224])
Class labels extracted mapping: {'nok': 0, 'ok': 1}


In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder(root='./FOLDER2/train', transform=transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
valid_dataset = datasets.ImageFolder(root='./FOLDER2/valid', transform=transform)
valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False)
test_dataset = datasets.ImageFolder(root='./FOLDER2/test', transform=transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
# 1. List the image files
image_files = os.listdir(images_path)
print(f"Found {len(image_files)} images.")

# 2. Grab the first image file name
sample_img_name = image_files[111]
sample_img_path = os.path.join(images_path, sample_img_name)

# 3. Load and display it
img = Image.open(sample_img_path)
plt.imshow(img)
plt.title(f"File: {sample_img_name}")
plt.axis('off')
plt.show()

Found 4900 images.


<Figure size 640x480 with 1 Axes>

In [ ]:
import random

def show_random_images(dataset, title, num_images=10):
    # Get random indices
    indices = random.sample(range(len(dataset)), num_images)

    plt.figure(figsize=(15, 6))
    plt.suptitle(title, fontsize=16)

    for i, idx in enumerate(indices):
        # Get image path and label index directly from dataset samples
        img_path, label_idx = dataset.samples[idx]
        label_name = dataset.classes[label_idx]

        # Load original image to avoid normalization artifacts
        img = Image.open(img_path)

        plt.subplot(2, 5, i + 1)
        plt.imshow(img)
        plt.title(f"{label_name}\n({os.path.basename(img_path)})")
        plt.axis('off')

    plt.tight_layout()
    plt.show()

# Show 10 random images from the training set
show_random_images(train_dataset, "10 Random Images from Training Set")

# Show 10 random images from the test set
show_random_images(test_dataset, "10 Random Images from Test Set")

<Figure size 1500x600 with 10 Axes>

<Figure size 1500x600 with 10 Axes>

# 2. Architecture Definitions

### 1. Basic CNN Architecture
Defining a simple convolutional neural network as a baseline.

In [ ]:
class BasicCNN(nn.Module):
    def __init__(self, num_classes=2):
        super(BasicCNN, self).__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # Output: 8 x 112 x 112

            # Block 2
            nn.Conv2d(8, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # Output: 16 x 56 x 56

            # Block 3
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)  # Output: 32 x 28 x 28
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 28 * 28, 32),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(32, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# Quick shape test
if __name__ == "__main__":
    model_basic = BasicCNN()
    dummy_input = torch.randn(32, 3, 224, 224)
    print(f"Basic CNN Output Shape: {model_basic(dummy_input).shape}") # Expected: [32, 2]

Basic CNN Output Shape: torch.Size([32, 2])


### 2. Base ResNet Architecture
A custom implementation of the Residual Network using identity and projection shortcuts.

In [ ]:
class ResNetBlock(nn.Module):
    def __init__(self, in_channels, filters, downsample=False):
        super().__init__()
        self.filters = filters
        self.downsample = downsample
        self.stride = 2 if downsample else 1

        # First convolutional layer (handles downsampling stride)
        self.conv1 = nn.Conv2d(in_channels, filters, kernel_size=3, padding=1, stride=self.stride, bias=False)
        self.bn1 = nn.BatchNorm2d(filters)
        self.relu = nn.ReLU()

        # Second convolutional layer
        self.conv2 = nn.Conv2d(filters, filters, kernel_size=3, padding=1, stride=1, bias=False)
        self.bn2 = nn.BatchNorm2d(filters)

        # Projection shortcut matching your logic (kernel_size=3)
        if self.downsample or in_channels != self.filters:
            self.projection = nn.Sequential(
                nn.Conv2d(in_channels, filters, kernel_size=3, padding=1, stride=self.stride, bias=False),
                nn.BatchNorm2d(filters)  # Standard practice to include BN after projection conv
            )
        else:
            self.projection = nn.Identity()

    def forward(self, x):
        # Main path
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        # Shortcut path
        shortcut = self.projection(x)

        # Residual connection + activation
        out += shortcut
        return self.relu(out)

class CustomBaseResNet(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        # Initial convolution block to process the raw input image
        self.init_conv = nn.Sequential(
            nn.Conv2d(3, 8, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(8),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        )

        # Stacking your custom ResNet blocks
        # Layer 1: No downsampling, channels remain 8
        self.layer1 = ResNetBlock(in_channels=8, filters=8, downsample=False)

        # Layer 2: Downsamples spatial dimensions by half, doubles channels to 16
        self.layer2 = ResNetBlock(in_channels=8, filters=16, downsample=True)

        # Layer 3: Downsamples spatial dimensions by half, doubles channels to 32
        self.layer3 = ResNetBlock(in_channels=16, filters=32, downsample=True)

        # Classifier head
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(32, num_classes)

    def forward(self, x):
        x = self.init_conv(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)

        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

# Quick shape test to verify it works with your dataloader
if __name__ == "__main__":
    model = CustomBaseResNet(num_classes=2)
    dummy_input = torch.randn(32, 3, 224, 224)
    output = model(dummy_input)
    print(f"ResNet Output Shape: {output.shape}") # Expected: torch.Size([32, 2])

ResNet Output Shape: torch.Size([32, 2])


### 3. SE-ResNet Architecture
Enhancing the ResNet with Squeeze-and-Excitation blocks for channel-wise feature recalibration.

In [ ]:
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        # Squeeze-and-Excitation bottleneck structure
        self.fc1 = nn.Conv2d(channels, channels // reduction, kernel_size=1)
        self.relu = nn.ReLU()
        self.fc2 = nn.Conv2d(channels // reduction, channels, kernel_size=1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # 1. Squeeze: Global Average Pooling across spatial dimensions
        w = x.mean(dim=(2, 3), keepdim=True)
        # 2. Excitation: Fully connected bottleneck layers
        w = self.fc1(w)
        w = self.relu(w)
        w = self.fc2(w)
        w = self.sigmoid(w)
        # 3. Scale: Recalibrate channel-wise feature responses
        return x * w

class SE_ResNetBlock(nn.Module):
    def __init__(self, in_channels, filters, downsample=False, reduction=16):
        super().__init__()
        self.filters = filters
        self.downsample = downsample
        self.stride = 2 if downsample else 1

        # Main Path
        self.conv1 = nn.Conv2d(in_channels, filters, kernel_size=3, padding=1, stride=self.stride, bias=False)
        self.bn1 = nn.BatchNorm2d(filters)
        self.relu = nn.ReLU()

        self.conv2 = nn.Conv2d(filters, filters, kernel_size=3, padding=1, stride=1, bias=False)
        self.bn2 = nn.BatchNorm2d(filters)

        # Squeeze-and-Excitation Block added right after the second batch normalization
        self.se = SEBlock(filters, reduction=reduction)

        # Shortcut Path
        if self.downsample or in_channels != self.filters:
            self.projection = nn.Sequential(
                nn.Conv2d(in_channels, filters, kernel_size=3, padding=1, stride=self.stride, bias=False),
                nn.BatchNorm2d(filters)
            )
        else:
            self.projection = nn.Identity()

    def forward(self, x):
        # Main path sequence
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        # Recalibrate features with SE block before residual addition
        out = self.se(out)

        # Residual connection
        shortcut = self.projection(x)
        out += shortcut
        return self.relu(out)

In [ ]:
class SE_ResNet(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        # Initial standard convolution block to process the raw input image
        self.init_conv = nn.Sequential(
            nn.Conv2d(3, 8, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(8),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        )

        # Stacking your custom SE-ResNet blocks - Note reduction=4 as channels are scaled down
        self.layer1 = SE_ResNetBlock(in_channels=8, filters=8, downsample=False, reduction=4)
        self.layer2 = SE_ResNetBlock(in_channels=8, filters=16, downsample=True, reduction=4)
        self.layer3 = SE_ResNetBlock(in_channels=16, filters=32, downsample=True, reduction=4)

        # Classifier head
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(32, num_classes)

    def forward(self, x):
        x = self.init_conv(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)

        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

# 3. The Training Engine

In [ ]:
def train_model(model, model_name, train_loader, valid_loader, criterion, optimizer, num_epochs=5, device='cuda', scheduler=None):
    print(f"--- Starting Training for {model_name} ---")

    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss': [], 'val_acc': []
    }

    best_model_wts = copy.deepcopy(model.state_dict())
    best_val_loss = float('inf') # Track lowest validation loss

    start_time = time.time()

    for epoch in range(num_epochs):
        # ---------- Training Phase ----------
        model.train()
        running_loss, correct, total = 0.0, 0, 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

        epoch_train_loss = running_loss / total
        epoch_train_acc = (correct / total) * 100

        # ---------- Validation Phase ----------
        model.eval()
        val_running_loss, val_correct, val_total = 0.0, 0, 0

        with torch.no_grad():
            for images, labels in valid_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)

                val_running_loss += loss.item() * images.size(0)
                _, predicted = outputs.max(1)
                val_total += labels.size(0)
                val_correct += predicted.eq(labels).sum().item()

        epoch_val_loss = val_running_loss / val_total
        epoch_val_acc = (val_correct / val_total) * 100

        # Save metrics
        history['train_loss'].append(epoch_train_loss)
        history['train_acc'].append(epoch_train_acc)
        history['val_loss'].append(epoch_val_loss)
        history['val_acc'].append(epoch_val_acc)

        # Get current LR
        current_lr = optimizer.param_groups[0]['lr']

        print(f"Epoch [{epoch+1}/{num_epochs}] LR: {current_lr:.6f} | "
              f"Train Loss: {epoch_train_loss:.4f}, Acc: {epoch_train_acc:.2f}% | "
              f"Val Loss: {epoch_val_loss:.4f}, Acc: {epoch_val_acc:.2f}%")

        if scheduler is not None:
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(epoch_val_loss)
            else:
                scheduler.step()

        # Save best model based on validation loss
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            best_model_wts = copy.deepcopy(model.state_dict())
            torch.save(model.state_dict(), f'best_{model_name}.pth')

    time_elapsed = time.time() - start_time
    print(f"Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s")
    print(f"Best Val Loss: {best_val_loss:.4f}\n")

    # Load best weights before returning
    model.load_state_dict(best_model_wts)
    return model, history

# 4. Execution Pipeline

In [ ]:
from torch.optim.lr_scheduler import ReduceLROnPlateau

# Initialization
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = nn.CrossEntropyLoss()
num_epochs = 30 # Adjust based on your computational limits

# 1. Initialize Models
model_cnn = BasicCNN(num_classes=2).to(device)
model_resnet = CustomBaseResNet(num_classes=2).to(device)
model_seresnet = SE_ResNet(num_classes=2).to(device)

# 2. Initialize Optimizers
opt_cnn = torch.optim.Adam(model_cnn.parameters(), lr=2e-3, weight_decay=1e-4)
opt_resnet = torch.optim.Adam(model_resnet.parameters(), lr=2e-3, weight_decay=1e-4)
opt_seresnet = torch.optim.Adam(model_seresnet.parameters(), lr=2e-3, weight_decay=1e-4)

# 3. Initialize Schedulers
sch_cnn = ReduceLROnPlateau(opt_cnn, mode='min', factor=0.5, patience=1)
sch_resnet = ReduceLROnPlateau(opt_resnet, mode='min', factor=0.5, patience=1)
sch_seresnet = ReduceLROnPlateau(opt_seresnet, mode='min', factor=0.5, patience=1)

# 4. Train Models and Store Histories
print("Starting sequential training pipeline...\n")

Starting sequential training pipeline...



In [ ]:
model_cnn, hist_cnn = train_model(model_cnn, "BasicCNN", train_loader, valid_loader, criterion, opt_cnn, num_epochs, device, scheduler=sch_cnn)

--- Starting Training for BasicCNN ---
Epoch [1/30] LR: 0.002000 | Train Loss: 0.5441, Acc: 66.11% | Val Loss: 0.4277, Acc: 84.67%
Epoch [2/30] LR: 0.002000 | Train Loss: 0.4260, Acc: 76.04% | Val Loss: 0.5017, Acc: 73.14%
Epoch [3/30] LR: 0.002000 | Train Loss: 0.3466, Acc: 82.12% | Val Loss: 0.2993, Acc: 84.39%
Epoch [4/30] LR: 0.002000 | Train Loss: 0.2628, Acc: 88.21% | Val Loss: 0.1699, Acc: 95.36%
Epoch [5/30] LR: 0.002000 | Train Loss: 0.2012, Acc: 92.27% | Val Loss: 0.0855, Acc: 99.16%
Epoch [6/30] LR: 0.002000 | Train Loss: 0.1506, Acc: 95.32% | Val Loss: 0.0512, Acc: 99.72%
Epoch [7/30] LR: 0.002000 | Train Loss: 0.1731, Acc: 94.47% | Val Loss: 0.2556, Acc: 94.23%
Epoch [8/30] LR: 0.002000 | Train Loss: 0.1465, Acc: 95.19% | Val Loss: 0.0899, Acc: 97.75%
Epoch [9/30] LR: 0.001000 | Train Loss: 0.0938, Acc: 97.63% | Val Loss: 0.1906, Acc: 93.25%
Epoch [10/30] LR: 0.001000 | Train Loss: 0.0888, Acc: 97.80% | Val Loss: 0.0701, Acc: 97.47%
Epoch [11/30] LR: 0.000500 | Train Loss:

In [ ]:
model_resnet, hist_resnet = train_model(model_resnet, "BaseResNet", train_loader, valid_loader, criterion, opt_resnet, num_epochs, device, scheduler=sch_resnet)

--- Starting Training for BaseResNet ---
Epoch [1/30] LR: 0.002000 | Train Loss: 0.3694, Acc: 83.98% | Val Loss: 0.2614, Acc: 83.97%
Epoch [2/30] LR: 0.002000 | Train Loss: 0.2324, Acc: 91.34% | Val Loss: 0.1904, Acc: 95.92%
Epoch [3/30] LR: 0.002000 | Train Loss: 0.1969, Acc: 92.30% | Val Loss: 0.0711, Acc: 98.31%
Epoch [4/30] LR: 0.002000 | Train Loss: 0.1113, Acc: 96.32% | Val Loss: 0.1607, Acc: 94.66%
Epoch [5/30] LR: 0.002000 | Train Loss: 0.1084, Acc: 96.25% | Val Loss: 0.0797, Acc: 98.73%
Epoch [6/30] LR: 0.001000 | Train Loss: 0.0954, Acc: 96.67% | Val Loss: 0.1144, Acc: 95.78%
Epoch [7/30] LR: 0.001000 | Train Loss: 0.0757, Acc: 97.63% | Val Loss: 0.1370, Acc: 95.64%
Epoch [8/30] LR: 0.000500 | Train Loss: 0.0593, Acc: 98.28% | Val Loss: 0.0860, Acc: 96.77%
Epoch [9/30] LR: 0.000500 | Train Loss: 0.0432, Acc: 98.87% | Val Loss: 0.0909, Acc: 96.62%
Epoch [10/30] LR: 0.000250 | Train Loss: 0.0451, Acc: 98.59% | Val Loss: 0.0443, Acc: 98.45%
Epoch [11/30] LR: 0.000250 | Train Los

In [ ]:
model_seresnet, hist_seresnet = train_model(model_seresnet, "SE_ResNet", train_loader, valid_loader, criterion, opt_seresnet, num_epochs, device, scheduler=sch_seresnet)

--- Starting Training for SE_ResNet ---
Epoch [1/30] LR: 0.002000 | Train Loss: 0.4053, Acc: 83.81% | Val Loss: 3.3645, Acc: 46.69%
Epoch [2/30] LR: 0.002000 | Train Loss: 0.2076, Acc: 92.13% | Val Loss: 0.9117, Acc: 51.62%
Epoch [3/30] LR: 0.002000 | Train Loss: 0.1927, Acc: 91.68% | Val Loss: 0.2182, Acc: 92.12%
Epoch [4/30] LR: 0.002000 | Train Loss: 0.1194, Acc: 96.01% | Val Loss: 0.0361, Acc: 99.44%
Epoch [5/30] LR: 0.002000 | Train Loss: 0.1045, Acc: 96.12% | Val Loss: 0.1584, Acc: 93.11%
Epoch [6/30] LR: 0.002000 | Train Loss: 0.0744, Acc: 97.39% | Val Loss: 0.0175, Acc: 99.72%
Epoch [7/30] LR: 0.002000 | Train Loss: 0.0678, Acc: 97.66% | Val Loss: 0.8784, Acc: 68.35%
Epoch [8/30] LR: 0.002000 | Train Loss: 0.0621, Acc: 97.66% | Val Loss: 0.0302, Acc: 99.16%
Epoch [9/30] LR: 0.001000 | Train Loss: 0.0360, Acc: 98.90% | Val Loss: 0.0688, Acc: 96.48%
Epoch [10/30] LR: 0.001000 | Train Loss: 0.0373, Acc: 98.38% | Val Loss: 0.1123, Acc: 94.94%
Epoch [11/30] LR: 0.000500 | Train Loss

# 5. Visual Comparison & Evaluation

In [ ]:
import random
import torch
from PIL import Image
import matplotlib.pyplot as plt

def display_model_predictions(test_ds, models_dict, device, num_images=10):
    # Pick 10 random indices
    indices = random.sample(range(len(test_ds)), num_images)

    plt.figure(figsize=(20, 10))
    plt.suptitle("Model Predictions vs True Labels (Test Set)", fontsize=18)

    for i, idx in enumerate(indices):
        # Get the tensor for prediction (normalized)
        tensor_img, label_idx = test_ds[idx]
        tensor_img = tensor_img.unsqueeze(0).to(device)

        # Get the original image for plotting (unnormalized)
        img_path, _ = test_ds.samples[idx]
        img = Image.open(img_path)
        true_label = test_ds.classes[label_idx]

        # Get predictions from each model
        preds = {}
        for name, model in models_dict.items():
            model.eval()
            with torch.no_grad():
                output = model(tensor_img)
                pred_idx = output.argmax(1).item()
                preds[name] = test_ds.classes[pred_idx]

        # Plot the image
        ax = plt.subplot(2, 5, i + 1)
        plt.imshow(img)

        # Format the title with True Label and Predictions
        title_text = f"True: {true_label}\n"
        title_text += f"CNN: {preds['Basic CNN']}\n"
        title_text += f"ResNet: {preds['Base ResNet']}\n"
        title_text += f"SE-ResNet: {preds['SE-ResNet']}"

        # Determine text color based on if any prediction is wrong
        all_correct = all(p == true_label for p in preds.values())
        text_color = 'green' if all_correct else 'red'

        plt.title(title_text, fontsize=11, color=text_color, loc='left')
        plt.axis('off')

    plt.tight_layout()
    plt.show()

# Create a dictionary of our trained models
trained_models_dict = {
    'Basic CNN': model_cnn,
    'Base ResNet': model_resnet,
    'SE-ResNet': model_seresnet
}

# Ensure device is set
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Display the predictions
display_model_predictions(test_dataset, trained_models_dict, device)

<Figure size 2000x1000 with 10 Axes>

In [ ]:
def show_misclassified_images(test_ds, models_dict, device, max_images=10):
    misclassified = []

    # Iterate through the test set to find misclassified examples
    for idx in range(len(test_ds)):
        tensor_img, label_idx = test_ds[idx]
        tensor_img = tensor_img.unsqueeze(0).to(device)
        true_label = test_ds.classes[label_idx]

        preds = {}
        is_wrong = False
        for name, model in models_dict.items():
            model.eval()
            with torch.no_grad():
                output = model(tensor_img)
                pred_idx = output.argmax(1).item()
                pred_label = test_ds.classes[pred_idx]
                preds[name] = pred_label
                if pred_label != true_label:
                    is_wrong = True

        if is_wrong:
            misclassified.append((idx, true_label, preds))

        if len(misclassified) >= max_images:
            break

    if not misclassified:
        print("No misclassified images found! The models performed perfectly on the test set.")
        return

    plt.figure(figsize=(20, 10))
    plt.suptitle("Misclassified Images (by at least one model)", fontsize=18)

    for i, (idx, true_label, preds) in enumerate(misclassified):
        img_path, _ = test_ds.samples[idx]
        img = Image.open(img_path)

        ax = plt.subplot(2, 5, i + 1)
        plt.imshow(img)

        # Format the title
        title_text = f"True: {true_label}\n"
        for name, pred in preds.items():
            title_text += f"{name}: {pred}\n"

        plt.title(title_text, fontsize=11, color='red', loc='left')
        plt.axis('off')

    plt.tight_layout()
    plt.show()

# Display the misclassified images
show_misclassified_images(test_dataset, trained_models_dict, device)

<Figure size 2000x1000 with 10 Axes>

In [ ]:
def get_metrics(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            outputs = model(images.to(device))
            preds = outputs.argmax(1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    # Get weighted averages
    p, r, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='weighted')
    return p, r, f1

# Collect data
models = {
    'Basic CNN': model_cnn,
    'Base ResNet': model_resnet,
    'SE-ResNet': model_seresnet
}

results = []
for name, m in models.items():
    p, r, f1 = get_metrics(m, test_loader, device)
    results.append({'Model': name, 'Metric': 'Precision', 'Score': p})
    results.append({'Model': name, 'Metric': 'Recall', 'Score': r})
    results.append({'Model': name, 'Metric': 'F1-Score', 'Score': f1})

df_results = pd.DataFrame(results)

# Visualization
plt.figure(figsize=(12, 6))
sns.barplot(data=df_results, x='Model', y='Score', hue='Metric', palette='magma')
plt.title('Comprehensive Test Set Comparison (Weighted Metrics)')
plt.ylim(0.8, 1.02)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

# Display the table
display(df_results)

<Figure size 1200x600 with 1 Axes>

,Model,Metric,Score
0,Basic CNN,Precision,0.985820
1,Basic CNN,Recall,0.985484
2,Basic CNN,F1-Score,0.985430
3,Base ResNet,Precision,0.996791
4,Base ResNet,Recall,0.996774
5,Base ResNet,F1-Score,0.996772
6,SE-ResNet,Precision,0.990473
7,SE-ResNet,Recall,0.990323
8,SE-ResNet,F1-Score,0.990299


In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("=== Model Architectures and Parameter Counts ===\n")
for name, model in trained_models_dict.items():
    print(f"Model: {name}")
    print(f"Total Trainable Parameters: {count_parameters(model):,}")
    print("-" * 50)
    print(model)
    print("=" * 80 + "\n")

=== Model Architectures and Parameter Counts ===

Model: Basic CNN
Total Trainable Parameters: 808,946
--------------------------------------------------
BasicCNN(
  (features): Sequential(
    (0): Conv2d(3, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(8, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=25088, out_features=32, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.5, inplace=False)
    (4): Linear(in_features=32, out_features=2, bias=True)
  )
)

Model: Base ResNet
Total Train

# 6. Final Conclusions

![w8_md032_006.png](../assets/w8_md032_006.png)

![w8_md033_007.png](../assets/w8_md033_007.png)

![w8_md034_008.png](../assets/w8_md034_008.png)

![w8_md036_009.png](../assets/w8_md036_009.png)